# Signpost-D, part 3: a JevBench dry run (public items only)

[JevBench](https://benchmarkheaven.com/jev-models) (Benchmark Heaven) scores decision models on typed decisions: accuracy, calibration, latency and cost, over easy, standard, hard and judge tiers, part of them held out. This notebook runs **only its public items** (`datasets/public/`: original, easy, hard), so the numbers are a local estimate, not a board score; nothing is submitted.

For each model in `MODELS`:
1. serve it with stock vLLM (`/v1/systemone`), as the published model card says;
2. run JevBench's own CLI with its `typesafe` adapter on every public item, then its `summarize`;
3. estimate the public-split Intelligence (chance-corrected, tier-weighted), as answered and with every yes/no made decisive (JevBench counts a yes/no whose P(yes) is between 0.2 and 0.8 as an abstention);
4. break the results down by task family and by input length, to show where the model fails.

## Settings

In [1]:
import os

env = os.environ.get
MODELS = env("MODELS", "skamalj/signpost-d-9b,skamalj/signpost-d-4b").split(",")
JEVBENCH_COMMIT = env("JEVBENCH_COMMIT", "609a664e81f46e03731600bd7bd9f1b7a61257c5")   # pinned: comparable runs
SERVE_FLAGS = env("SERVE_FLAGS", "--max-model-len 32768 --max-num-seqs 8 --logprobs-mode raw_logprobs")
TP = int(env("TP", "1"))                         # GPUs to split the model across (2 on Kaggle's two T4s)
PORT = 8000
NOUL_DECISIVE = 0.8                              # P(yes) inside (0.2, 0.8) counts as an abstention
TIERS = {"easy": 0.1, "original": 0.2, "hard": 0.4}   # public file -> the board's tier weight
RUNS_DIR = os.path.abspath("jevbench_runs")
print(MODELS, JEVBENCH_COMMIT[:7], SERVE_FLAGS, f"TP {TP}")

['skamalj/signpost-d-9b', 'skamalj/signpost-d-4b'] 609a664 --max-model-len 32768 --max-num-seqs 8 --logprobs-mode raw_logprobs TP 1


## vLLM (in its own environment) and JevBench's code at the pinned commit

In [2]:
!pip install -q uv requests

import json, math, statistics, subprocess, sys, time
from pathlib import Path

import requests

VLLM_ENV = Path("vllm-env").absolute()
if not (VLLM_ENV / "bin" / "vllm").exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "-q", str(VLLM_ENV), "--python", sys.executable], check=True)
    subprocess.run([sys.executable, "-m", "uv", "pip", "install", "-q", "--python", str(VLLM_ENV / "bin" / "python"),
                    "-U", "vllm", "--torch-backend=auto", "--extra-index-url", "https://wheels.vllm.ai/nightly"], check=True)
VLLM_VERSION = subprocess.run([str(VLLM_ENV / "bin" / "python"), "-c", "import vllm; print(vllm.__version__)"],
                              capture_output=True, text=True).stdout.strip()

JEVBENCH_DIR = os.path.abspath("jevbench")
if not os.path.isdir(JEVBENCH_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/fstandhartinger/jevbench.git", JEVBENCH_DIR], check=True)
subprocess.run(["git", "-C", JEVBENCH_DIR, "fetch", "-q", "origin"], check=False)
subprocess.run(["git", "-C", JEVBENCH_DIR, "checkout", "-q", JEVBENCH_COMMIT], check=True)
for req in ("requirements.txt",):
    if os.path.exists(f"{JEVBENCH_DIR}/{req}"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{JEVBENCH_DIR}/{req}"], check=False)
TASK_FILES = {tier: f"{JEVBENCH_DIR}/datasets/public/{tier}.jsonl" for tier in TIERS}
RAW = {tier: [json.loads(line) for line in open(path)] for tier, path in TASK_FILES.items()}
GPU = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(f"vLLM {VLLM_VERSION} | JevBench at {JEVBENCH_COMMIT[:7]} | "
      + " | ".join(f"{t} {len(v)}" for t, v in RAW.items()) + f" public items | {GPU}")
print("item fields:", sorted(RAW["hard"][0].keys()))


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


vLLM 0.34.0rc1.dev572+g0d9d953d6 | JevBench at 609a664 | easy 48 | original 72 | hard 111 public items | NVIDIA L40S
item fields: ['expected', 'family', 'group', 'id', 'labels', 'provenance', 'question', 'split', 'state']


## Task families and input length

JevBench items carry their family (routing, policy checks, severity scores, …) in their metadata; the field name is found from the items themselves. Input length is the state's length in characters.

In [3]:
def family_of(item):
    for key in ("family", "task_family", "category", "domain", "source"):
        if key in item and isinstance(item[key], str):
            return item[key]
    meta = item.get("meta") or item.get("metadata") or {}
    for key in ("family", "task_family", "category", "domain", "source"):
        if key in meta and isinstance(meta[key], str):
            return meta[key]
    return "?"


def length_of(item):
    state = item.get("state", item.get("input", ""))
    return len(state if isinstance(state, str) else json.dumps(state))


ITEM = {it["id"]: {"tier": tier, "family": family_of(it), "chars": length_of(it),
                   "type": (it.get("question") or {}).get("type", "?")} for tier, items in RAW.items() for it in items}
print("families:", dict(sorted({f: sum(1 for v in ITEM.values() if v["family"] == f)
                                for f in {v["family"] for v in ITEM.values()}}.items())))
print("chars: median", int(statistics.median(v["chars"] for v in ITEM.values())),
      "| max", max(v["chars"] for v in ITEM.values()))

families: {'adequacy': 12, 'adversarial': 6, 'ambiguous': 7, 'extraction': 24, 'fact': 12, 'intent': 24, 'judge_hard': 17, 'long_policy': 19, 'multi_hop': 18, 'ordinal': 12, 'policy': 12, 'probability': 10, 'routing': 12, 'routing_hard': 5, 'temporal_numeric': 15, 'tool_selection': 12, 'tradeoff': 6, 'trap': 8}
chars: median 128 | max 14986


## Serving one model

In [4]:
def start_server(model):
    log_path = Path(RUNS_DIR) / f"vllm-{model.split('/')[-1]}.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = [str(VLLM_ENV / "bin" / "vllm"), "serve", model, "--port", str(PORT), "--tensor-parallel-size", str(TP),
           *SERVE_FLAGS.split()]
    proc = subprocess.Popen(cmd, stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
    t0 = time.time()
    while time.time() - t0 < 1800:
        try:
            if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=5).ok:
                print(f"vLLM ready for {model} in {(time.time() - t0) / 60:.1f} min", flush=True)
                return proc
        except requests.RequestException:
            pass
        if proc.poll() is not None:
            break
        time.sleep(10)
    print(open(log_path).read()[-3000:])
    raise RuntimeError(f"vLLM did not start for {model}")


def stop_server(proc):
    proc.terminate()
    try:
        proc.wait(timeout=120)
    except subprocess.TimeoutExpired:
        proc.kill()
    time.sleep(10)

## JevBench's CLI and summariser

In [5]:
def cli_run(model):
    run_dir = f"{RUNS_DIR}/{model.split('/')[-1]}"
    os.makedirs(run_dir, exist_ok=True)
    all_tasks = ",".join(TASK_FILES.values())
    cmd = [sys.executable, "-m", "jevbench.cli", "run", "--tasks", all_tasks, "--adapter", "typesafe",
           "--endpoint", f"http://127.0.0.1:{PORT}", "--model", model, "--key-env", "",
           "--results", f"{run_dir}/results.jsonl", "--ledger", f"{run_dir}/ledger.jsonl",
           "--raw-dir", f"{run_dir}/raw", "--reserve-usd", "0", "--price-in-per-m", "0",
           "--price-out-per-m", "0", "--cost-basis", "local_gpu_no_billable_account",
           "--run-label", model.split("/")[-1], "--manifest", f"{run_dir}/manifest.json"]
    t0 = time.perf_counter()
    done = subprocess.run(cmd, cwd=JEVBENCH_DIR, capture_output=True, text=True)
    print(done.stdout[-1500:], done.stderr[-1500:])
    print(f"CLI run: exit {done.returncode} in {(time.perf_counter() - t0) / 60:.1f} min -> {run_dir}", flush=True)
    return run_dir


def cli_summary(task_files, results):
    out = subprocess.run([sys.executable, "-m", "jevbench.cli", "summarize", "--tasks", task_files, "--results", results],
                         cwd=JEVBENCH_DIR, capture_output=True, text=True, check=True)
    return json.loads(out.stdout)


def summaries_for(run_dir):
    out = {"all": cli_summary(",".join(TASK_FILES.values()), f"{run_dir}/results.jsonl")}
    lines = open(f"{run_dir}/results.jsonl").read().splitlines()
    for tier, path in TASK_FILES.items():
        wanted = {json.loads(line)["id"] for line in open(path)}
        part = f"{run_dir}/results-{tier}.jsonl"           # the summariser wants one file's items
        with open(part, "w") as fh:
            fh.writelines(line + "\n" for line in lines if json.loads(line)["task_id"] in wanted)
        out[tier] = cli_summary(path, part)
    return out

## The Intelligence estimate (public items, tier-weighted)

Per tier and question type: choice and yes/no are chance-corrected accuracy; score is the reduction of the expected-level error against always answering the middle. A yes/no whose P(yes) is between 0.2 and 0.8 counts as wrong (an abstention), as on the board; the second line makes every yes/no decisive, to show how much the abstentions cost.

In [6]:
sys.path.insert(0, JEVBENCH_DIR)
from jevbench.tasks import load_jsonl

TASKS = {tier: load_jsonl(path) for tier, path in TASK_FILES.items()}
BY_ID = {t.id: t for ts in TASKS.values() for t in ts}


def competence(rs, kind, decisive_all):
    ts = [BY_ID[r["task_id"]] for r in rs]
    if kind == "choice":
        acc = statistics.mean(bool(r["correct"]) for r in rs)
        base = statistics.mean(1 / len(t.labels) for t in ts)
        return 100 * (acc - base) / (1 - base), None
    if kind == "noul":
        hits, decisive = [], []
        for r in rs:
            p = (r.get("probs") or {}).get("yes")
            sure = p is not None and (decisive_all or p >= NOUL_DECISIVE or p <= 1 - NOUL_DECISIVE)
            decisive.append(sure)
            hits.append(sure and bool(r["correct"]))
        return 100 * (statistics.mean(hits) - 0.5) / 0.5, statistics.mean(decisive)
    errs, base = [], []
    for r, t in zip(rs, ts):
        mid = (len(t.labels) - 1) / 2
        ev = r.get("ordinal_ev")
        errs.append(abs(ev - t.expected) if ev is not None else abs(len(t.labels) - 1))
        base.append(abs(mid - t.expected))
    if statistics.mean(base) == 0:
        base = [statistics.mean(abs(i - t.expected) for i in range(len(t.labels))) for t in ts]
    return 100 * (statistics.mean(base) - statistics.mean(errs)) / statistics.mean(base), None


def intelligence(records, decisive_all):
    rows, total = {}, 0.0
    for tier, weight in TIERS.items():
        per = {}
        for kind in ("choice", "noul", "score"):
            rs = [r for r in records if ITEM[r["task_id"]]["tier"] == tier and BY_ID[r["task_id"]].question["type"] == kind]
            if rs:
                per[kind] = competence(rs, kind, decisive_all)
        o1s = min(100.0, max(0.0, statistics.mean(c for c, _ in per.values())))
        rows[tier] = (per, o1s)
        total += weight * o1s
    return rows, total / sum(TIERS.values())

## Every model

In [7]:
REPORT = {}
for model in MODELS:
    print(f"\n========== {model}", flush=True)
    proc = start_server(model)
    try:
        run_dir = cli_run(model)
    finally:
        stop_server(proc)
    summ = summaries_for(run_dir)
    records = [json.loads(line) for line in open(f"{run_dir}/results.jsonl")]
    fmt = lambda v, d=3: "-" if v is None else f"{v:.{d}f}"
    print(f"{'':9s}{'n':>5s} {'acc':>7s} {'brier':>7s} {'ece':>7s} {'ord MAE':>8s} {'valid':>7s} {'p50 ms':>7s}")
    for name, s in summ.items():
        print(f"{name:9s}{s['n_scorable']:>5d} {fmt(s['accuracy']):>7s} {fmt(s['brier_mean']):>7s} "
              f"{fmt(s['ece']['ece']):>7s} {fmt(s['ordinal_mae']):>8s} {fmt(s['schema_validity']):>7s} "
              f"{s['latency']['p50_s'] * 1000:>7.0f}")
    intel = {}
    for label, decisive_all in (("as answered", False), ("yes/no made decisive", True)):
        rows, value = intelligence(records, decisive_all)
        intel[label] = value
        print(f"{label}: Intelligence (public, tier-weighted) {value:.1f}")
        for tier, (per, o1s) in rows.items():
            parts = " | ".join(f"{k} {c:.1f}" + (f" (decisive {d:.0%})" if d is not None else "") for k, (c, d) in per.items())
            print(f"  {tier:9s} {o1s:5.1f} | {parts}")
    fam = {}
    for r in records:
        f = ITEM[r["task_id"]]["family"]
        fam.setdefault(f, []).append(bool(r.get("correct")))
    print("by family (accuracy, n):", {f: (round(statistics.mean(v), 3), len(v)) for f, v in sorted(fam.items())})
    buckets = [(0, 2000), (2000, 6000), (6000, 15000), (15000, 10**9)]
    by_len = {}
    for lo, hi in buckets:
        v = [bool(r.get("correct")) for r in records if lo <= ITEM[r["task_id"]]["chars"] < hi]
        if v:
            by_len[f"{lo}-{hi if hi < 10**9 else 'more'} chars"] = (round(statistics.mean(v), 3), len(v))
    print("by input length (accuracy, n):", by_len)
    REPORT[model] = {"summary": summ, "intelligence": intel, "by_family": fam and
                     {f: [round(statistics.mean(v), 4), len(v)] for f, v in fam.items()}, "by_length": by_len}

json.dump(REPORT, open(f"{RUNS_DIR}/report.json", "w"), indent=1)
print("\nRESULT", json.dumps({m: {"accuracy": r["summary"]["all"]["accuracy"], **r["intelligence"]} for m, r in REPORT.items()}))


========== skamalj/signpost-d-9b


vLLM ready for skamalj/signpost-d-9b in 4.8 min


[jevbench] adapter=typesafe endpoint=http://127.0.0.1:8000 tasks=231 cap=$15.0 ledger=/__modal/volumes/vo-VRQjOFsLyhxyrzyS9Kgv3t/jevbench-dry/jevbench_runs/signpost-d-9b/ledger.jsonl
10/231 completed
20/231 completed
30/231 completed
40/231 completed
50/231 completed
60/231 completed
70/231 completed
80/231 completed
90/231 completed
100/231 completed
110/231 completed
120/231 completed
130/231 completed
140/231 completed
150/231 completed
160/231 completed
170/231 completed
180/231 completed
190/231 completed
200/231 completed
210/231 completed
220/231 completed
230/231 completed
[jevbench] done: 231/231 attempted, 0 failed; charged=$0.0000
 
CLI run: exit 0 in 0.4 min -> /__modal/volumes/vo-VRQjOFsLyhxyrzyS9Kgv3t/jevbench-dry/jevbench_runs/signpost-d-9b


             n     acc   brier     ece  ord MAE   valid  p50 ms
all        231   0.831   0.228   0.071    0.346   1.000      57
easy        48   1.000   0.003   0.038        -   1.000      55
original    72   0.931   0.101   0.140    0.129   1.000      55
hard       111   0.694   0.408   0.070    0.779   1.000      91
as answered: Intelligence (public, tier-weighted) 35.9
  easy      100.0 | choice 100.0 | noul 100.0 (decisive 100%)
  original   68.4 | choice 93.0 | noul 25.0 (decisive 62%) | score 87.1
  hard        3.6 | choice 60.1 | noul -15.8 (decisive 53%) | score -33.6
yes/no made decisive: Intelligence (public, tier-weighted) 52.6
  easy      100.0 | choice 100.0 | noul 100.0 (decisive 100%)
  original   85.0 | choice 93.0 | noul 75.0 (decisive 100%) | score 87.1
  hard       24.6 | choice 60.1 | noul 47.4 (decisive 100%) | score -33.6
by family (accuracy, n): {'adequacy': (0.833, 12), 'adversarial': (1, 6), 'ambiguous': (0.857, 7), 'extraction': (1, 24), 'fact': (1, 12), 'inte

vLLM ready for skamalj/signpost-d-4b in 1.8 min


[jevbench] adapter=typesafe endpoint=http://127.0.0.1:8000 tasks=231 cap=$15.0 ledger=/__modal/volumes/vo-VRQjOFsLyhxyrzyS9Kgv3t/jevbench-dry/jevbench_runs/signpost-d-4b/ledger.jsonl
10/231 completed
20/231 completed
30/231 completed
40/231 completed
50/231 completed
60/231 completed
70/231 completed
80/231 completed
90/231 completed
100/231 completed
110/231 completed
120/231 completed
130/231 completed
140/231 completed
150/231 completed
160/231 completed
170/231 completed
180/231 completed
190/231 completed
200/231 completed
210/231 completed
220/231 completed
230/231 completed
[jevbench] done: 231/231 attempted, 0 failed; charged=$0.0000
 
CLI run: exit 0 in 0.3 min -> /__modal/volumes/vo-VRQjOFsLyhxyrzyS9Kgv3t/jevbench-dry/jevbench_runs/signpost-d-4b


             n     acc   brier     ece  ord MAE   valid  p50 ms
all        231   0.801   0.268   0.066    0.353   1.000      56
easy        48   1.000   0.009   0.050        -   1.000      55
original    72   0.903   0.160   0.138    0.135   1.000      54
hard       111   0.649   0.450   0.074    0.787   1.000      81
as answered: Intelligence (public, tier-weighted) 29.5
  easy       91.7 | choice 100.0 | noul 83.3 (decisive 92%)
  original   57.5 | choice 86.0 | noul 0.0 (decisive 50%) | score 86.5
  hard        0.0 | choice 60.1 | noul -26.3 (decisive 45%) | score -35.0
yes/no made decisive: Intelligence (public, tier-weighted) 47.7
  easy      100.0 | choice 100.0 | noul 100.0 (decisive 100%)
  original   82.5 | choice 86.0 | noul 75.0 (decisive 100%) | score 86.5
  hard       17.2 | choice 60.1 | noul 26.3 (decisive 100%) | score -35.0
by family (accuracy, n): {'adequacy': (0.917, 12), 'adversarial': (1, 6), 'ambiguous': (0.857, 7), 'extraction': (1, 24), 'fact': (1, 12), 'intent'